# Unit 4 solutions: hold the speed, follow the line

These reference solutions predict each answer first where the lesson gives a law (the drag balance, the steady circle, the error's rate), and then measure it on racekit's kinematic car, advanced exactly as the runner advances it.
Every run is deterministic (jitter 0) or seeded (seeds 0–4), so every number below is reproducible, and executable asserts check each answer against an independent derivation or the lesson's own numbers.
Every number is a number about the **kinematic car**: a model, not the truth.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit.cars import KinematicCar
from racekit.measure import noise_aware_better, repeat_races, spread
from racekit.runner import CONTROL_DT, run_race
from racekit.tracks import load_track
from racekit.variants import write_variant

AGENTS = Path("../../projects/racekit/agents")
UNIT_04 = AGENTS / "unit-04.py"
DT = CONTROL_DT  # 0.05 s: one control tick; advance the car with 10 substeps per tick, as the runner does

# racekit's kinematic car (a model, not the truth)
WHEELBASE = 2.875  # m, L
DELTA_MAX = math.radians(70.0)  # wheel angle at |steer| = 1; delta = -steer * DELTA_MAX (a positive steer turns right)
A_THROTTLE = 6.0  # m/s^2 at full throttle
C_DRAG = 4.0e-4  # 1/m: drag deceleration C_DRAG * v * |v|


def wrap_angle(angle):
    # Unit 2: wrap an angle in radians to [-pi, pi)
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

## Exercise 1 - The P loop's error, predicted and measured

In the steady state v̇ = 0, so A_THROTTLE·kp·e = C_DRAG·(target − e)², the quadratic C·e² − (2C·v* + a_T·kp)·e + C·v*² = 0, and the P loop settles at its smaller root.
120 s is many time constants for every case, so the simulation must sit on the root.
For a 0.01 m/s error at 20 m/s, solve the same balance for kp: kp = C·(20 − 0.01)² / (a_T · 0.01) ≈ 2.66.

In [ ]:
def equilibrium_error(kp, target):
    # The smaller root of C e^2 - (2 C v* + a_T kp) e + C v*^2 = 0.
    b = 2.0 * C_DRAG * target + A_THROTTLE * kp
    return (b - math.sqrt(b * b - 4.0 * C_DRAG * C_DRAG * target * target)) / (2.0 * C_DRAG)


def p_loop_final_error(kp, target, seconds):
    car = KinematicCar()  # jitter 0
    car.reset(0.0, 0.0, 0.0, 0.0)
    for _ in range(int(round(seconds / DT))):
        error = target - car.v
        throttle = min(max(kp * error, 0.0), 1.0)
        brake = min(max(-kp * error, 0.0), 1.0)
        car.advance(throttle, 0.0, brake, 0.0, 0, DT, 10)
    return target - car.v


CASES = [(0.05, 20.0), (0.2, 20.0), (1.0, 20.0), (0.05, 30.0)]
predicted = {case: equilibrium_error(*case) for case in CASES}
simulated = {case: p_loop_final_error(case[0], case[1], 120.0) for case in CASES}
# A_T kp e = C (v* - e)^2 with e = 0.01 and v* = 20, solved for kp
kp_for_one_cm = C_DRAG * (20.0 - 0.01) ** 2 / (A_THROTTLE * 0.01)
why_never_zero = (
    "In the steady state the throttle kp * e must balance the drag C_DRAG * (target - e)^2, which is "
    "positive at any speed above zero, so the error e must be positive for every finite kp: "
    "zero error would mean zero throttle, and zero throttle cannot hold any speed against drag."
)

for case in CASES:
    print(f"kp {case[0]:4}, target {case[1]:4}: predicted {predicted[case]:.6f} m/s, after 120 s {simulated[case]:.6f} m/s")
print(f"kp for a 1 cm/s error at 20 m/s: {kp_for_one_cm:.4f}")

# checks: the root satisfies the drag balance, matches the simulation, and the lesson's starter number
for kp, target in CASES:
    e = predicted[(kp, target)]
    assert abs(A_THROTTLE * kp * e - C_DRAG * (target - e) ** 2) < 1e-12
    assert 0.0 < e < target
    assert abs(simulated[(kp, target)] - e) < 1e-4
assert abs(equilibrium_error(0.05, 20.0) - 0.5067) < 1e-4  # the lesson's number
assert abs(equilibrium_error(kp_for_one_cm, 20.0) - 0.01) < 1e-9
assert predicted[(0.05, 30.0)] > predicted[(0.05, 20.0)]  # more drag at 30 m/s, a larger error

## Exercise 2 - Windup, and conditional integration

The class follows the lesson's description: the candidate integral and the P + I command with it decide whether the integral moves (the D term is added after the decision, as in the lesson's class), and `update` returns the unclipped command.
From 10 m/s the P term alone asks for 0.5 · 20 = 10, so the throttle saturates at once and the plain integral winds up to about 34.5 m; with conditional integration it stops near the 0.64 m that the steady state at 30 m/s needs (c·v²/a_T = 0.06, an integral of 0.6 m with K_I = 0.1).

In [ ]:
class PID:
    # u = kp * e + ki * (integral of e) + kd * (rate of e); conditional integration with limits.
    def __init__(self, kp, ki=0.0, kd=0.0, limits=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.limits = limits
        self.reset()

    def reset(self):
        self.integral = 0.0
        self.previous = None

    def update(self, error, dt, rate=None):
        candidate = self.integral + error * dt
        if self.limits is None:
            self.integral = candidate
        else:
            low, high = self.limits
            command = self.kp * error + self.ki * candidate
            inside = low < command < high
            unwinding = (command >= high and error < 0) or (command <= low and error > 0)
            if inside or unwinding:
                self.integral = candidate
        if rate is None:
            rate = (error - self.previous) / dt if self.previous is not None else 0.0
        self.previous = error
        return self.kp * error + self.ki * self.integral + self.kd * rate


def step_response(controller, v0, target, seconds):
    car = KinematicCar()  # jitter 0
    car.reset(0.0, 0.0, 0.0, v0)
    ticks = int(round(seconds / DT))
    speeds, integrals = np.zeros(ticks), np.zeros(ticks)
    for n in range(ticks):
        command = controller.update(target - car.v, DT)
        throttle = min(max(command, 0.0), 1.0)
        brake = min(max(-command, 0.0), 1.0)
        car.advance(throttle, 0.0, brake, 0.0, 0, DT, 10)
        speeds[n] = car.v
        integrals[n] = controller.integral
    return speeds, integrals


def settling_from_mask(outside):
    # The time of the first tick from which no later tick is outside; tick n ends at (n + 1) * DT.
    indices = np.nonzero(outside)[0]
    if len(indices) == 0:
        return 0.0
    if indices[-1] == len(outside) - 1:
        return math.inf
    return float((indices[-1] + 2) * DT)


def step_metrics(speeds, v0, target):
    step = target - v0
    overshoot = max(0.0, float((np.max(speeds) - target) / step * 100.0))
    settling = settling_from_mask(np.abs(speeds - target) > 0.02 * step)
    return overshoot, settling, float(target - speeds[-1])


speeds_plain, integral_plain = step_response(PID(0.5, 0.1), 10.0, 30.0, 40.0)
speeds_conditional, integral_conditional = step_response(PID(0.5, 0.1, limits=(0.0, 1.0)), 10.0, 30.0, 40.0)
metrics_plain = step_metrics(speeds_plain, 10.0, 30.0)
metrics_conditional = step_metrics(speeds_conditional, 10.0, 30.0)
peak_integral_plain = float(np.max(integral_plain))
peak_integral_conditional = float(np.max(integral_conditional))

t = DT * np.arange(1, len(speeds_plain) + 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, speeds_plain, label="PI (0.5, 0.1), no anti-windup")
ax.plot(t, speeds_conditional, label="PI (0.5, 0.1), conditional integration")
ax.axhline(30.0, color="grey", linestyle="--", linewidth=1.0, label="target 30 m/s")
ax.set_xlabel("t (s)")
ax.set_ylabel("speed (m/s)")
ax.set_title("Step response from 10 to 30 m/s (the kinematic car, jitter 0)")
ax.legend()
plt.show()

what_windup_is = (
    "In the plain run the throttle was saturated at 1 for seconds while the error was large, and the "
    f"integral kept adding that error (it reached {peak_integral_plain:.1f} m) although the controller could "
    "not push any harder; at 30 m/s the I term alone still asked for more than full throttle, so the speed "
    "had to overshoot until the negative error unwound the integral."
)

for name, metrics, peak in (("plain", metrics_plain, peak_integral_plain),
                            ("conditional", metrics_conditional, peak_integral_conditional)):
    print(f"{name:>12}: overshoot {metrics[0]:.4f}%, settling {metrics[1]:.2f} s, error {metrics[2]:+.6f} m/s, "
          f"peak integral {peak:.4f} m")

# checks: the PID matches the lesson's numbers on the lesson's step, and anti-windup does its job here
assert abs(step_metrics(step_response(PID(0.5, 0.1), 0.0, 20.0, 30.0)[0], 0.0, 20.0)[0] - 28.71) < 0.01
lesson_clamped = step_metrics(step_response(PID(0.5, 0.1, limits=(0.0, 1.0)), 0.0, 20.0, 30.0)[0], 0.0, 20.0)
assert abs(lesson_clamped[0] - 0.26) < 0.01 and abs(lesson_clamped[1] - 3.55) < 1e-9
assert metrics_plain[0] > 5.0 * metrics_conditional[0] and metrics_plain[1] > metrics_conditional[1]
assert peak_integral_plain > 10.0 * peak_integral_conditional
# the D term: finite difference 0 on the first call, then (e - previous) / dt; a passed rate wins
probe = PID(0.0, kd=1.0)
assert probe.update(1.0, 0.5) == 0.0 and probe.update(2.0, 0.5) == 2.0 and probe.update(5.0, 0.5, rate=-1.0) == -1.0
probe.reset()
assert probe.integral == 0.0 and probe.previous is None

## Exercise 3 - Tune by step response

Every pair is measured on both steps with conditional integration on (0, 1); the rule is applied by filtering, then sorting on (larger settling time, kp, ki).
The big step cannot settle before full throttle alone brings the car into the ±0.4 m/s band, about 3.3 s from rest, so the stiff pairs all crowd against that floor.
The small step (2 m/s, which saturates the throttle only briefly) is the one that separates the gains.

In [ ]:
KPS = (0.25, 0.5, 1.0, 2.0)
KIS = (0.05, 0.1, 0.2, 0.4)
grid = {}
for kp in KPS:
    for ki in KIS:
        big = step_metrics(step_response(PID(kp, ki, limits=(0.0, 1.0)), 0.0, 20.0, 30.0)[0], 0.0, 20.0)
        small = step_metrics(step_response(PID(kp, ki, limits=(0.0, 1.0)), 18.0, 20.0, 30.0)[0], 18.0, 20.0)
        grid[(kp, ki)] = (big, small)


def acceptable(pair):
    return all(metrics[0] <= 1.0 and abs(metrics[2]) <= 0.002 for metrics in grid[pair])


candidates = [pair for pair in grid if acceptable(pair)]
best = min(candidates, key=lambda pair: (max(grid[pair][0][1], grid[pair][1][1]), pair[0], pair[1]))
best_settling = max(grid[best][0][1], grid[best][1][1])
big_settlings = [grid[pair][0][1] for pair in grid]
big_step_range = (min(big_settlings), max(big_settlings))
saturation_note = (
    "From rest the P term alone asks for a command of 5 to 40, so the throttle is pinned at 1 for about "
    "3 s whatever the gains are and the car accelerates at its limit: the big step mostly measures the car "
    "(every pair with kp >= 0.5 settles in 3.30-3.55 s), and only the soft kp = 0.25 rows, whose I term "
    "overshoots after the throttle unsaturates, take longer."
)

print("  kp    ki | big: overshoot, settling, error | small: overshoot, settling, error")
for (kp, ki), (big, small) in grid.items():
    mark = " <- best" if (kp, ki) == best else (" ok" if acceptable((kp, ki)) else "")
    print(f"{kp:4} {ki:5} | {big[0]:6.3f}% {big[1]:6.2f} s {big[2]:+.5f} | "
          f"{small[0]:6.3f}% {small[1]:6.2f} s {small[2]:+.5f}{mark}")
print(f"best {best}, larger settling {best_settling:.2f} s; big-step settling range {big_step_range}")

# checks: the rule, re-applied by brute force over every pair
assert len(grid) == 16 and best in candidates
for pair in candidates:
    key = (max(grid[pair][0][1], grid[pair][1][1]), pair[0], pair[1])
    assert key >= (best_settling, best[0], best[1])
assert grid[(0.5, 0.1)][0][:2] == lesson_clamped[:2]  # the lesson's anti-windup row
# no controller can settle the big step before full throttle brings the car into the band
car = KinematicCar()
car.reset(0.0, 0.0, 0.0, 0.0)
ticks_at_full_throttle = 0
while car.v < 20.0 - 0.02 * 20.0:
    car.advance(1.0, 0.0, 0.0, 0.0, 0, DT, 10)
    ticks_at_full_throttle += 1
assert big_step_range[0] >= ticks_at_full_throttle * DT - 1e-9
assert all(3.25 < grid[(kp, ki)][0][1] < 3.6 for kp in KPS if kp >= 0.5 for ki in KIS)
print(f"full throttle alone reaches the band after {ticks_at_full_throttle * DT:.2f} s")

## Exercise 4 - The heading error is the D term

With x' = v cos ψ and y' = v sin ψ, ė = −sin θ·v cos ψ + cos θ·v sin ψ = v sin(ψ − θ) = −v sin ψ_e.
The drag-balancing throttle with the car's current v makes v̇ exactly 0 at every RK4 stage, so v stays 20 m/s.
Without the heading term the law is a spring with no damper (the lesson's §5), so the error never settles; the damping grows with K_HEAD until the car stops crossing the line.
The settling times use Exercise 2's rule on the error recorded at the start of each tick.

In [ ]:
THETA = math.pi / 6.0
K_HEADS = (0.0, 0.25, 0.5, 1.0, 2.0)
K_E = 0.2
LINE_SPEED = 20.0
why_e_dot = (
    "Differentiating e = -sin(theta) x + cos(theta) y with x' = v cos(psi) and y' = v sin(psi) gives "
    "e' = v (sin(psi) cos(theta) - cos(psi) sin(theta)) = v sin(psi - theta) = -v sin(psi_e)."
)


def line_drive(k_head, seconds=15.0):
    # Hold the line through the origin at heading THETA; return e, v and psi_e at the start of every tick.
    car = KinematicCar()
    car.reset(-1.5 * math.sin(THETA), 1.5 * math.cos(THETA), THETA, LINE_SPEED)
    ticks = int(round(seconds / DT))
    errors, speeds, heading_errors = np.zeros(ticks), np.zeros(ticks), np.zeros(ticks)
    for n in range(ticks):
        e = -math.sin(THETA) * car.x + math.cos(THETA) * car.y
        psi_e = wrap_angle(THETA - car.yaw)
        errors[n], speeds[n], heading_errors[n] = e, car.v, psi_e
        delta = -K_E * e + k_head * psi_e
        steer = min(max(-delta / DELTA_MAX, -1.0), 1.0)
        car.advance(C_DRAG * car.v * car.v / A_THROTTLE, steer, 0.0, 0.0, 0, DT, 10)
    return errors, speeds, heading_errors


lateral = {}
LINE_RUNS = {}
for k_head in K_HEADS:
    errors, speeds, heading_errors = line_drive(k_head)
    LINE_RUNS[k_head] = (errors, speeds, heading_errors)
    lateral[k_head] = (max(0.0, -float(np.min(errors))), settling_from_mask(np.abs(errors) > 0.05))
smallest_damped_k_head = min(k for k in K_HEADS if lateral[k][0] <= 0.05 and lateral[k][1] < math.inf)

errors, speeds, heading_errors = LINE_RUNS[1.0]
central = (errors[2:] - errors[:-2]) / (2.0 * DT)
rate_gap = float(np.max(np.abs(central + speeds[1:-1] * np.sin(heading_errors[1:-1]))))
k_d_at_20 = 1.0 / 20.0
k_d_at_30 = 1.0 / 30.0

for k_head in K_HEADS:
    overshoot, settling = lateral[k_head]
    print(f"K_HEAD {k_head:4}: overshoot {overshoot:.4f} m, settling {settling:.2f} s")
print(f"smallest damped K_HEAD {smallest_damped_k_head}; rate gap {rate_gap:.3e} m/s; K_D {k_d_at_20} s at 20 m/s, {k_d_at_30:.4f} s at 30 m/s")

# checks
assert abs(LINE_RUNS[0.0][0][0] - 1.5) < 1e-12  # the start is 1.5 m left of the line
assert lateral[0.0][1] == math.inf  # P alone never settles
# the measured rate is the error's rate: the central difference's truncation error is largest on the first
# ticks, where the wheel angle changes fastest (under 6% of the peak rate), and below 2 mm/s after 1 s
gaps = np.abs(central + speeds[1:-1] * np.sin(heading_errors[1:-1]))
peak_rate = float(np.max(np.abs(speeds * np.sin(heading_errors))))
assert rate_gap == float(np.max(gaps)) < 0.06 * peak_rate
assert np.max(gaps[20:]) < 0.002
assert np.max(np.abs(speeds - LINE_SPEED)) < 1e-9  # the drag-balancing throttle holds the speed
assert abs(k_d_at_20 / k_d_at_30 - 1.5) < 1e-12

## Exercise 5 - Hold a circle, with and without the feedforward

In the steady state ψ_e ≈ 0 (the kinematic bicycle's heading is tangent to its rear-axle path), so the P term alone must supply the circle's wheel angle: −K_E·e = arctan(L / R), e = −arctan(L / R) / K_E, negative: the car rides outside the turn.
The check solves the exact balance −K_E·e = arctan(L / (R − e)) (the car drives the circle of radius R − e) by iteration.
On a circle of radius R, three points 2 m apart by arc length are (R sin(s/R), R − R cos(s/R)) for s = 0, 2, 4; the Menger circle through three points of a circle is that circle, so κ = 1/40 up to rounding.

In [ ]:
RADII = (20.0, 40.0, 80.0)
K_HEAD = 1.0
CIRCLE_SPEED = 15.0


def circle_drive(radius, seconds, feedforward=False, steering=None):
    # Drive the left-turning circle centred at (0, R) from the origin heading +x.
    # steering(e, psi_e) returns the PD(+I) part of the wheel angle; default: -K_E e + K_HEAD psi_e.
    # Returns e after every advance.
    car = KinematicCar()
    car.reset(0.0, 0.0, 0.0, CIRCLE_SPEED)
    ticks = int(round(seconds / DT))
    errors = np.zeros(ticks)
    for n in range(ticks):
        e = radius - math.hypot(car.x, car.y - radius)
        theta = math.atan2(car.y - radius, car.x) + math.pi / 2.0
        psi_e = wrap_angle(theta - car.yaw)
        delta = -K_E * e + K_HEAD * psi_e if steering is None else steering(e, psi_e)
        if feedforward:
            delta += math.atan2(WHEELBASE / radius, 1.0)  # arctan(L kappa)
        steer = min(max(-delta / DELTA_MAX, -1.0), 1.0)
        car.advance(C_DRAG * car.v * car.v / A_THROTTLE, steer, 0.0, 0.0, 0, DT, 10)
        errors[n] = radius - math.hypot(car.x, car.y - radius)
    return errors


LAST_10_S = int(round(10.0 / DT))
predicted_no_ff = {radius: -math.atan2(WHEELBASE / radius, 1.0) / K_E for radius in RADII}
steady_no_ff = {radius: float(np.mean(circle_drive(radius, 30.0)[-LAST_10_S:])) for radius in RADII}
steady_ff = {radius: float(np.mean(circle_drive(radius, 30.0, feedforward=True)[-LAST_10_S:])) for radius in RADII}


def menger(ax_, ay, bx, by, cx, cy):
    # Unit 2: signed curvature of the circle through a, b, c, left positive
    cross = (bx - ax_) * (cy - ay) - (by - ay) * (cx - ax_)
    sides = math.hypot(bx - ax_, by - ay) * math.hypot(cx - bx, cy - by) * math.hypot(ax_ - cx, ay - cy)
    return 2.0 * cross / sides


arc = [(40.0 * math.sin(s / 40.0), 40.0 - 40.0 * math.cos(s / 40.0)) for s in (0.0, 2.0, 4.0)]
menger_40 = menger(*arc[0], *arc[1], *arc[2])
why_feedforward = (
    "The feedforward supplies the circle's own wheel angle arctan(L / R), which the PD terms can only "
    "produce from a standing offset e = -arctan(L / R) / K_E; the prediction is slightly off at R = 20 "
    "because the car then drives a circle of radius R - e, about 0.7 m wider, which needs a slightly "
    "smaller angle than arctan(L / R), so the measured offset is a little smaller."
)

for radius in RADII:
    print(f"R {radius:4}: predicted {predicted_no_ff[radius]:+.4f} m, no feedforward {steady_no_ff[radius]:+.4f} m, "
          f"feedforward {steady_ff[radius]:+.2e} m")
print(f"Menger curvature on R = 40: {menger_40:.12f} (1/40 = {1 / 40})")

# checks: the steady state is the circle of radius R - e driven with -K_E e, solved independently
for radius in RADII:
    # -K_E e = arctan(L / (R - e)): a fixed-point iteration from the prediction
    e = predicted_no_ff[radius]
    for _ in range(100):
        e = -math.atan2(WHEELBASE / (radius - e), 1.0) / K_E
    assert abs(steady_no_ff[radius] - e) < 1e-3
    assert abs(steady_ff[radius]) < 1e-3 < abs(steady_no_ff[radius])
    assert abs(steady_no_ff[radius] - predicted_no_ff[radius]) < 0.05
assert abs(steady_no_ff[20.0]) < abs(predicted_no_ff[20.0])  # the wider circle needs less angle
assert abs(menger_40 - 1.0 / 40.0) < 1e-12

## Exercise 6 - Tune K_E on the track, beyond the noise

Each gain is a real submission made by `write_variant` (K_E = 0.2 is the reference itself), raced five times on `oval` with jitter 0.05 and seeds 0–4.
Unit 1's rule decides "beyond the noise": the gap in medians must exceed both ranges.

In [ ]:
GAINS = (0.1, 0.2, 0.4)
variant_folder = Path(mkdtemp())


def jittered():
    return KinematicCar(jitter=0.05)


def rms_offset(result):
    offset = result.telemetry["offset"][result.telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))


rms_medians, rms_ranges, time_medians = {}, {}, {}
all_clean = True
for k_e in GAINS:
    variant = write_variant(UNIT_04, variant_folder / f"k_e_{k_e}.py", {"K_E": repr(k_e)})
    runs = repeat_races(variant, "oval", n=5, car_factory=jittered)
    rms = [rms_offset(result) for result in runs.results]
    rms_medians[k_e] = float(np.median(rms))
    rms_ranges[k_e] = spread(rms)
    time_medians[k_e] = runs.median
    all_clean = all_clean and runs.all_finished and runs.collisions == 0
    print(f"K_E {k_e}: RMS {np.round(rms, 4)}, median {rms_medians[k_e]:.4f} m (range {rms_ranges[k_e]:.4f}); "
          f"time median {runs.median:.2f} s; collisions {runs.collisions}")
best_k_e = min(GAINS, key=lambda k_e: rms_medians[k_e])
beats_reference = bool(
    best_k_e != 0.2
    and rms_medians[0.2] - rms_medians[best_k_e] > max(rms_ranges[best_k_e], rms_ranges[0.2])
)
same_time = len(set(time_medians.values())) == 1
verdict = (
    f"Not on this evidence alone: K_E = {best_k_e} sits closer to the line on oval "
    f"({'beyond' if beats_reference else 'not beyond'} the noise), but the race time "
    f"{'is the same for all three gains' if same_time else 'barely moves'}, so the gain buys accuracy, not time; "
    "first I would measure the other two tracks, the largest offset and the steering's weave, and the behaviour "
    "at higher speeds and with actuator delay, where a stiffer P term with the same damping is the first to oscillate."
)
print(f"best K_E {best_k_e}, beats the reference beyond the noise: {beats_reference}; all clean: {all_clean}")
print(verdict)

# checks
assert all_clean
assert abs(rms_medians[0.2] - 0.1041) < 5e-4  # the lesson's reference median (the variant is the reference)
assert abs(time_medians[0.2] - 216.40) < 0.05 + 1e-9
assert best_k_e == min(rms_medians, key=rms_medians.get)
assert rms_medians[0.4] < rms_medians[0.2] < rms_medians[0.1]  # a stiffer P term holds the line closer here

## Exercise 7 - Challenge: the derivative kick in the speed loop

On tick 400 the error jumps by 5 m/s, so the finite difference reads 5 / 0.05 = 100 m/s² and the D term adds kd · 100 = 5 to the command; the measured-speed rate does not see the jump (the car's speed did not change).
But the P term alone already asks for 0.5 · 5 = 2.5 there, so the throttle is clipped at 1 in both versions, and the kick changes nothing the car feels.

In [ ]:
KICK_TICK = 400
TICKS_7 = 800
kick, peak_command, second_step = {}, {}, {}
for version in ("difference", "measured"):
    controller = PID(0.5, 0.1, 0.05, limits=(0.0, 1.0))
    car = KinematicCar()
    car.reset(0.0, 0.0, 0.0, 0.0)
    starts, speeds, d_terms, commands = np.zeros(TICKS_7), np.zeros(TICKS_7), np.zeros(TICKS_7), np.zeros(TICKS_7)
    for n in range(TICKS_7):
        target = 25.0 if n >= KICK_TICK else 20.0
        starts[n] = car.v
        error = target - car.v
        if version == "difference":
            rate = (error - controller.previous) / DT if controller.previous is not None else 0.0
            command = controller.update(error, DT)
        else:
            rate = -(starts[n] - starts[n - 1]) / DT if n > 0 else 0.0
            command = controller.update(error, DT, rate=rate)
        d_terms[n], commands[n] = controller.kd * rate, command
        car.advance(min(max(command, 0.0), 1.0), 0.0, min(max(-command, 0.0), 1.0), 0.0, 0, DT, 10)
        speeds[n] = car.v
    kick[version] = float(d_terms[KICK_TICK])
    peak_command[version] = float(np.max(commands[KICK_TICK:]))
    second_step[version] = step_metrics(speeds[KICK_TICK:], starts[KICK_TICK], 25.0)

kick_verdict = (
    f"The finite difference kicks by {kick['difference']:.2f} on tick 400, but the throttle is already "
    "clipped at 1 by the P term's 2.5, so the kick never reaches the car and both versions take the same "
    "second step; in a steering loop the wheel angle is not saturated by the jump, so the kick goes "
    "straight to the wheels as a one-tick swerve (the lesson's 69 degrees)."
)
for version in kick:
    print(f"{version:>10}: kick {kick[version]:+.4f}, peak command {peak_command[version]:.4f}, second step {second_step[version]}")
print(kick_verdict)

# checks: the kick is kd * (the 5 m/s jump plus the tick's error change) / DT; the measured rate has no jump
assert abs(kick["difference"] - 0.05 * 5.0 / DT) < 0.01
assert abs(kick["measured"]) < 0.01
assert peak_command["difference"] > peak_command["measured"] > 1.0
assert all(abs(a - b) < 0.05 for a, b in zip(second_step["difference"], second_step["measured"]))

## Exercise 8 - Challenge: an I term on the steering instead of the feedforward

The I term is the controller's integral of −e, so `PID(K_E, ki=K_I, kd=K_HEAD)` with `rate=ψ_e` returns exactly −K_E·e + K_I·∫(−e)dt + K_HEAD·ψ_e.
The error is recorded after each advance (as in Exercise 5), and the settling time uses Exercise 2's rule.
The corners are maximal non-cyclic runs of |κ| > 0.005; the mean waypoint spacing is the lap length over the number of waypoints (about 2 m).

In [ ]:
K_IS = (0.02, 0.05, 0.1)
CIRCLE_R = 40.0


def with_integral(k_i):
    controller = PID(K_E, ki=k_i, kd=K_HEAD)

    def steering(e, psi_e):
        return controller.update(-e, DT, rate=psi_e)

    return steering


steady_i, settle_i = {}, {}
for k_i in K_IS:
    errors = circle_drive(CIRCLE_R, 60.0, steering=with_integral(k_i))
    steady_i[k_i] = float(np.mean(errors[-LAST_10_S:]))
    settle_i[k_i] = settling_from_mask(np.abs(errors) > 0.02)
ff_errors = circle_drive(CIRCLE_R, 60.0, feedforward=True)
settle_ff = settling_from_mask(np.abs(ff_errors) > 0.02)

track = load_track("test-circuit")
xs, ys = track.x, track.y
count = len(xs)
kappa = np.array([menger(xs[i - 1], ys[i - 1], xs[i], ys[i], xs[(i + 1) % count], ys[(i + 1) % count])
                  for i in range(count)])
in_corner = np.abs(kappa) > 0.005
runs, length = [], 0
for flag in in_corner:
    if flag:
        length += 1
    elif length:
        runs.append(length)
        length = 0
if length:
    runs.append(length)
spacing = track.length / count  # the mean waypoint spacing (the lap length over the waypoints)
corner_seconds = sorted(n * spacing / 20.0 for n in runs)
i_or_ff = (
    "The feedforward: it holds the circle from the first tick, while even the fastest I term here needs "
    f"{min(settle_i.values()):.1f} s to wind the steady offset away, longer than "
    f"{sum(seconds < min(settle_i.values()) for seconds in corner_seconds)} of the {len(corner_seconds)} corners "
    f"of test-circuit last at 20 m/s ({min(corner_seconds):.1f} to {max(corner_seconds):.1f} s), so an I term "
    "would spend most of every corner off the line, and its integral would carry each corner's offset into "
    "the next straight."
)

for k_i in K_IS:
    print(f"K_I {k_i}: steady error {steady_i[k_i]:+.2e} m, settling {settle_i[k_i]:.2f} s")
print(f"feedforward: settling {settle_ff:.2f} s")
print(f"{len(corner_seconds)} corners, durations (s): {np.round(corner_seconds, 2)}; spacing {spacing:.4f} m")
print(i_or_ff)

# checks
assert all(abs(steady_i[k_i]) < 0.02 for k_i in K_IS)  # the I term removes the steady offset
assert settle_ff < min(settle_i.values())
assert sum(runs) == int(np.sum(in_corner)) and all(n > 0 for n in runs)
assert abs(spacing - 2.0) < 0.2  # about 2 m between waypoints